In [6]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os
from dotenv import load_dotenv
load_dotenv()

DATA_LOC = os.getenv("DATA_LOC")
print(DATA_LOC)

~/Data/DCIA_DATA/Data/GENERAL SOURCES


# RMBW Data

What is RMBW?
- Regionale Monitor Brede Welvaart
- Maps the braod prosperity of municipalities, provinces and COROP regions using 42 indicators
- broad prosperity concenrs the quality of life here and now and the extent to which this comes at the expense of that of future generations or of people elsewhere in the world

In [7]:
rmbw_loc = os.path.join(DATA_LOC, "rmbw2025_resultaten.csv")
rmbw = pd.read_csv(rmbw_loc)

print(rmbw.columns)
rmbw.head(50)

Index(['ind_code', 'statcode', 'trend_periode', 'trend_kleur', 'positie',
       'pos_van_n', 'pos_jaar', 'pos_kleur', 'een_na_laatste_jaar',
       'laatste_jaar', 'mut_tekst_nl', 'mut_kleur'],
      dtype='str')


,ind_code,statcode,trend_periode,trend_kleur,positie,pos_van_n,pos_jaar,pos_kleur,een_na_laatste_jaar,laatste_jaar,mut_tekst_nl,mut_kleur
0,01.07_NW,NL01,2017-2024,groen,1,1,2024,grijs,2023.0,2024.0,"-6,6%",rood
1,01.07_NW,PV22,2017-2024,groen,5,12,2024,grijs,2023.0,2024.0,"-3,8%",rood
2,01.07_NW,PV24,2017-2024,groen,6,12,2024,grijs,2023.0,2024.0,"-4,2%",rood
3,01.07_NW,PV21,2017-2024,groen,8,12,2024,grijs,2023.0,2024.0,"-4,4%",rood
4,01.07_NW,PV25,2017-2024,groen,3,12,2024,groen,2023.0,2024.0,"-4,1%",rood
5,01.07_NW,PV20,2017-2024,groen,12,12,2024,rood,2023.0,2024.0,"-9,4%",rood
6,01.07_NW,PV31,2017-2024,groen,9,12,2024,grijs,2023.0,2024.0,"-10,1%",rood
7,01.07_NW,PV30,2017-2024,groen,1,12,2024,groen,2023.0,2024.0,"-4,8%",rood
8,01.07_NW,PV27,2017-2024,groen,10,12,2024,rood,2023.0,2024.0,"-12,1%",rood
9,01.07_NW,PV23,2017-2024,groen,7,12,2024,grijs,2023.0,2024.0,"-2,4%",rood


In [8]:
print(rmbw.shape)

(12343, 12)


Columns
- ind_code: Looks like an indicator code
- statcode: city code or province code. From the translation city code would make sense, but looking at examples it seems to be region
- trend_periode: trend period
- 'trend_kleur: trend color
- 'positie': position
- 'pos_van_n': position out of n
- 'pos_jaar': position year
- 'pos_kleur: position colour
- 'een_na_laatste_jaar: one before last year (used to compute the change)
- 'laatste_jaar': last year
- 'mut_tekst_nl': mutation? text, looks like percentage change
- 'mut_kleur': mutation? color



In [9]:
print(rmbw["statcode"].unique())

<StringArray>
[  'NL01',   'PV22',   'PV24',   'PV21',   'PV25',   'PV20',   'PV31',
   'PV30',   'PV27',   'PV23',
 ...
 'GM0355', 'GM0299', 'GM0637', 'GM0638', 'GM1892', 'GM0879', 'GM0301',
 'GM1896', 'GM0642', 'GM0193']
Length: 395, dtype: str


Stat code is either province or city.  We should be able to get province codes and municipality codes from here: https://www.cbs.nl/nl-nl/onze-diensten/methoden/classificaties/overig/gemeentelijke-indelingen-per-jaar/indeling-per-jaar/gemeentelijke-indeling-op-1-januari-2026
- NL01 is probably the whole of the country
- PV is province
- GM is gementee, so municipality

In [10]:
print(rmbw["ind_code"].unique())

<StringArray>
[   '01.07_NW',    '01.10_NW',    '01_GZ_08',    '01_GZ_20',  '01_KL_H_01',
  '01_KL_H_36', '01_KL_OK_08', '01_KL_OK_10', '01_KL_OK_11', '01_KL_OK_14',
 '01_KL_OK_30', '01_KL_OK_39', '01_KL_PK_01', '01_KL_PK_03', '01_KL_PK_07',
 '02_HB_MK_30', '02_HB_NK_30', '02_HB_NK_31', '02_HB_SK_01',   '03_PVI_04',
    '04_VE_03',   '05_OPL_08',   '07_K&E_03',   '09_E&V_03',   '09_E&V_12',
   '09_E&V_13',    '10.31_NW',    '15.15_NW',   '18_FIN_32',     'R_GZ_03',
  'R_HN_AV_02', 'R_HN_MIL_01', 'R_HN_MIL_02',  'R_HN_SL_02',  'R_HN_SL_03',
  'R_HN_SL_04',   'R_HN_V_01',   'R_L_NK_01',   'R_L_NK_02',   'R_L_NK_03',
   'R_L_NK_04',   'R_L_SK_01']
Length: 42, dtype: str


IND code stands for the indicator code. We have 42 separate indicators.

- description of the indicators: https://www.cbs.nl/nl-nl/visualisaties/regionale-monitor-brede-welvaart/beschrijving-indicatoren
- description of the indicator with the matching code is available in rmwb2025_meta.csv
- These 42 indicators seem crucial and very useful, from distance to green areas to OV, to cofees, green house gases, GDP. We should be able to build a ton of indicators around this data.

In [11]:
print(rmbw["trend_periode"].unique())

#Only have one period, 2017-2024

<StringArray>
['2017-2024']
Length: 1, dtype: str


This has only position from N no actual indicator values. What is the position compared with, the region?

In [12]:
rmbw["percentile"] = rmbw["positie"] / rmbw["pos_van_n"]

rmbw.head()

,ind_code,statcode,trend_periode,trend_kleur,positie,pos_van_n,pos_jaar,pos_kleur,een_na_laatste_jaar,laatste_jaar,mut_tekst_nl,mut_kleur,percentile
0,01.07_NW,NL01,2017-2024,groen,1,1,2024,grijs,2023.0,2024.0,"-6,6%",rood,1.000000
1,01.07_NW,PV22,2017-2024,groen,5,12,2024,grijs,2023.0,2024.0,"-3,8%",rood,0.416667
2,01.07_NW,PV24,2017-2024,groen,6,12,2024,grijs,2023.0,2024.0,"-4,2%",rood,0.500000
3,01.07_NW,PV21,2017-2024,groen,8,12,2024,grijs,2023.0,2024.0,"-4,4%",rood,0.666667
4,01.07_NW,PV25,2017-2024,groen,3,12,2024,groen,2023.0,2024.0,"-4,1%",rood,0.250000


Pivot table of municipality/province in the top percentile for each category

In [13]:
rmbw_pivot = rmbw.pivot_table(
    index="statcode",
    columns=["ind_code"],
    values="percentile",
    aggfunc="first" #This will only give one value so no need for calculating mean or any other agg func, I checked
)

rmbw_pivot.head()

ind_code,01.07_NW,01.10_NW,01_GZ_08,01_GZ_20,01_KL_H_01,01_KL_H_36,01_KL_OK_08,01_KL_OK_10,01_KL_OK_11,01_KL_OK_14,...,R_HN_MIL_02,R_HN_SL_02,R_HN_SL_03,R_HN_SL_04,R_HN_V_01,R_L_NK_01,R_L_NK_02,R_L_NK_03,R_L_NK_04,R_L_SK_01
statcode,,,,,,,,,,,,,,,,,,,,,
CR01,0.800,0.875,1.000,NaN,0.325,0.975,0.625,0.900,1.000,0.900,...,0.375,0.325,0.85,0.850,NaN,NaN,NaN,0.225,0.200,NaN
CR02,0.825,0.850,0.950,NaN,0.100,0.950,0.075,0.750,0.975,1.000,...,0.825,0.325,1.00,0.650,NaN,NaN,NaN,0.200,0.225,NaN
CR03,0.975,0.350,0.100,NaN,0.950,1.000,0.875,0.450,0.525,0.875,...,0.550,0.650,0.65,0.475,NaN,NaN,NaN,0.325,0.625,NaN
CR04,0.775,0.575,0.575,NaN,0.375,0.900,0.650,0.150,0.450,0.350,...,0.125,0.125,0.35,0.625,NaN,NaN,NaN,0.100,0.450,NaN
CR05,0.375,0.100,0.450,NaN,0.075,0.725,0.225,0.125,0.375,0.950,...,0.325,0.325,0.85,0.725,NaN,NaN,NaN,0.025,0.350,NaN


### The problem: no raw data only position out of N. I suggest we use the raw data directly
Download RMBW data from this page, select 2025 and data (instead of results): https://www.cbs.nl/nl-nl/visualisaties/regionale-monitor-brede-welvaart/downloads I added this to the onedrive


In [14]:
rmbw_data_loc = os.path.join(DATA_LOC, "rmbw2025_data.csv")
rmbw_data = pd.read_csv(rmbw_data_loc)

rmbw_data.head()

,indicator,statcode,jaar,waarde,ondermarge,bovenmarge
0,01_KL_H_01,NL01,2013,"83,6","82,6","84,5"
1,01_KL_H_01,NL01,2014,"84,6","83,7","85,5"
2,01_KL_H_01,NL01,2015,"83,9",83,"84,8"
3,01_KL_H_01,NL01,2016,"85,2","84,3","86,1"
4,01_KL_H_01,NL01,2017,"85,4","84,5","86,3"


3 values for indicator:
- waarde: value
- ondermarge: lower bond
- bovenmarge: upper bound

In [15]:
print(rmbw_data["jaar"].unique()) #many years are available, I suggest using all years since 2017 to match with the rmbw_resultaten data
rmbw_data = rmbw_data.loc[rmbw_data["jaar"] >= 2017]
#rmbw_data = rmbw_data.loc[rmbw_data["jaar"] == 2024]

[2013 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024 2011 2012
 1995 1996 1997 1998 1999 2000 2001 2002 2003 2004 2005 2006 2007 2008
 2009 2010]


In [16]:
#convert numerical values to not use , for separator
numerical_cols = ["waarde", "ondermarge", "bovenmarge"]

for col in numerical_cols:
    rmbw_data[col] = rmbw_data[col].str.replace(".", "") #remove dots
    rmbw_data[col] =  rmbw_data[col].str.replace(",", ".") #remove dots
    rmbw_data[col] = rmbw_data[col].astype(float)

In [17]:
rmbw_meta_loc = os.path.join(DATA_LOC, "rmbw2025_meta.csv")
rmbw_meta = pd.read_csv(rmbw_meta_loc)

rmbw_meta.head()



,ind_code,indicatornaam,meta_code,waarde
0,01.07_NW,Mediaan vermogen van huishoudens,alg_aspect1,Brede welvaart 'later'
1,01.07_NW,Mediaan vermogen van huishoudens,alg_thema1,Economisch kapitaal
2,01.07_NW,Mediaan vermogen van huishoudens,alg_sorteervolgorde,0
3,01.07_NW,Mediaan vermogen van huishoudens,alg_tekst_noot_dashboard,2024 cijfers zijn voorlopig en de correctie vo...
4,01.07_NW,Mediaan vermogen van huishoudens,tregio_afronding,-2


In [18]:
#for some reason meta features multiple rows for the same ind_code and indicatornaam with some meta_code that I am not sure where to match
#To avoid issues I will group by ind_code
rmbw_meta = rmbw_meta.groupby(by="ind_code")["indicatornaam"].first()

In [19]:
rmbw_data = pd.merge(rmbw_data, rmbw_meta, how="left", left_on="indicator", right_on="ind_code")

rmbw_data.head()

,indicator,statcode,jaar,waarde,ondermarge,bovenmarge,indicatornaam
0,01_KL_H_01,NL01,2017,85.4,84.5,86.3,Tevredenheid met het leven
1,01_KL_H_01,NL01,2018,85.7,84.8,86.6,Tevredenheid met het leven
2,01_KL_H_01,NL01,2019,87.3,86.5,88.2,Tevredenheid met het leven
3,01_KL_H_01,NL01,2020,84.8,83.9,85.8,Tevredenheid met het leven
4,01_KL_H_01,NL01,2021,83.6,82.6,84.7,Tevredenheid met het leven


In [20]:
rmbw_data_pivot = rmbw_data.pivot_table(
    index="statcode",
    columns="indicatornaam",
    values="waarde",
    aggfunc="mean" #mean of the last 3 years
)

print(rmbw_data_pivot.shape)
rmbw_data_pivot.head()

(395, 42)


indicatornaam,Aantal ondervonden delicten,Afstand tot basisschool,Afstand tot café e.d.,Afstand tot openbaar groen,Afstand tot ov,Afstand tot sportterrein,Arbeidsduur per week,Bebouwd gebied,Bevolking met een startkwalificatie,Broeikasgasemissies,...,Tevredenheid met het leven,Tevredenheid met sociaal leven,Tevredenheid met vrije tijd,Tevredenheid met woning,Tevredenheid met woonomgeving,Vaak onveilig voelen in de buurt,Vertrouwen in andere mensen,Vertrouwen in instituties,Vrijwilligerswerk,Werkloosheid
statcode,,,,,,,,,,,,,,,,,,,,,
CR01,NaN,0.9750,1.8875,0.5,0.5,1.0,27.0625,10.057143,65.8125,9.271429,...,84.3375,81.0625,77.8750,86.566667,84.566667,NaN,51.0625,52.5500,41.9750,4.6625
CR02,NaN,1.0750,1.6625,0.6,0.4,1.0,27.4500,10.157143,69.6625,53.614286,...,85.5625,80.8500,77.8000,78.266667,77.800000,NaN,55.9750,54.5000,47.4500,4.6000
CR03,NaN,0.8375,1.3750,0.5,0.4,1.1,26.6625,11.271429,78.5250,29.957143,...,81.2875,77.3750,73.3750,83.333333,82.266667,NaN,67.7125,63.3750,48.2375,5.4625
CR04,NaN,0.7875,1.8875,0.8,0.4,0.9,26.9625,8.671429,71.5375,9.042857,...,87.7000,80.8750,76.3125,85.166667,85.166667,NaN,67.6375,63.0250,52.0375,4.6750
CR05,NaN,0.9500,1.9750,0.7,0.5,1.0,27.4125,7.442857,71.8875,11.828571,...,87.2000,82.9500,79.4750,89.166667,87.533333,NaN,67.7375,62.6875,53.2000,4.0625


In [21]:
number_of_na = rmbw_data_pivot.isna().sum().sum() #We have a total of 2604 na
print(f"{round((number_of_na / (rmbw_data_pivot.shape[0] * rmbw_data_pivot.shape[1])) * 100, 2)}% of cells are NA")

25.66% of cells are NA


We get quite a lot of NAs, have to be careful about that.

### RMBW initial exploration summary:
- use the final resulting pivot talbe above to get an idea of how a municipality scores on the 42 indicators
- we need to be aware of the fact that there are a lot of NAs
- we are now using data from the last 8 years available (2017-2024). We can narrow this done if needed
- Data available both on the municipality, province and country level
- We can do comparisons to province or country